# 01 · Auditoría física, de esquema, categorías y granularidad

**Pregunta.** ¿Qué contienen realmente los 192 CSV oficiales y bajo qué condiciones pueden compararse los seis años?

**Entradas.** Manifiestos del notebook 00, seis diccionarios TXT y CSV extraídos. La unidad candidata se comprueba, no se presupone.

**Método.** Lectura completa y conservadora como texto mediante DuckDB; auditoría por archivo y año de formato, esquema, nulos, rangos, categorías, duplicados, claves, conciliación `LN = SV + NV + NS` y cobertura territorial. Los identificadores se mantienen como texto.

**Supuestos.** Un mismo código de sección entre años no prueba identidad territorial. `SEXO`, `EDAD` y las filas no se recodifican. Ausencia de categoría, valor faltante y conteo cero se reportan por separado.

**Salidas.** Inventario Parquet, matrices CSV, reporte HTML y evidencia para los documentos de definiciones y auditoría. No se construye panel ni target.

**Recursos esperados.** Lectura aproximada de 3.6 GB; varios minutos, memoria acotada por ejecución columnar.

## Entorno y entradas

Se detecta la raíz, se comprueba que el notebook 00 produjo los manifiestos y se mide tiempo/memoria del proceso actual.

In [ ]:
from pathlib import Path
import hashlib
import json
import resource
import time

import pandas as pd
from IPython.display import HTML, Markdown, display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "configs/sources.yaml").exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_STARTED = time.perf_counter()
required_inputs = [
    PROJECT_ROOT / "data/manifests/source_manifest.csv",
    PROJECT_ROOT / "data/manifests/zip_member_inventory.csv",
    PROJECT_ROOT / "data/interim/extracted",
]
input_checks = {path.relative_to(PROJECT_ROOT).as_posix(): path.exists() for path in required_inputs}
display(pd.Series(input_checks, name="existe").to_frame())
assert all(input_checks.values()), "Faltan entradas del notebook 00"


In [ ]:
display(Markdown(
    f"**Interpretación.** Las `{len(required_inputs)}` entradas declaradas existen. "
    "La auditoría comienza desde archivos persistidos, no desde variables de otro notebook."
))

## Documentación incluida en los ZIP

Se leen y hashean los seis diccionarios. Se comparan sus textos para detectar cambios documentales antes de interpretar columnas.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


dictionary_rows = []
dictionary_texts = {}
for year in [2009, 2012, 2015, 2018, 2021, 2024]:
    matches = list((PROJECT_ROOT / "data/interim/extracted" / str(year)).rglob("01_descripcioncampos*.txt"))
    assert len(matches) == 1, (year, matches)
    path = matches[0]
    text = path.read_text(encoding="utf-8-sig")
    dictionary_texts[year] = text
    dictionary_rows.append({
        "year": year,
        "path": path.relative_to(PROJECT_ROOT).as_posix(),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "mentions_derfe_2021": "DERFE 2021" in text,
        "mentions_derfe_2023": "DERFE 2023" in text,
        "mentions_non_binary": "No binario" in text,
    })
dictionary_inventory = pd.DataFrame(dictionary_rows)
display(dictionary_inventory)

In [ ]:
old_hashes = dictionary_inventory.loc[dictionary_inventory.year < 2024, "sha256"].nunique()
display(Markdown(
    f"**Hallazgo.** Los diccionarios 2009–2021 tienen `{old_hashes}` hash distinto(s): son "
    "binariamente idénticos y documentan distritos en el marco DERFE 2021. El diccionario 2024 "
    "cambia al marco DERFE 2023, agrega `SEXO=2` (No binario) y omite de la lista de NS el caso "
    "de cuadernillo solicitado por autoridad jurisdiccional. La sección se describe en todos los "
    "años como la del año correspondiente."
))

## Lectura completa e inventario físico

Se recorren los 192 CSV. Para preservar ceros y códigos, todas las columnas se leen inicialmente como texto; los rangos numéricos se calculan con conversiones explícitas que cuentan valores inválidos. El inventario por archivo incluye hash, formato, filas, columnas, nulos y muestra.

In [ ]:
from radar27.audit.raw_schema import audit_all

audit_started = time.perf_counter()
inventory, year_summary, categories, section_frames = audit_all(PROJECT_ROOT)
audit_seconds = time.perf_counter() - audit_started
peak_rss_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024

display(inventory[[
    "year", "path", "bytes", "encoding", "has_utf8_bom", "separator",
    "line_ending", "rows", "column_count", "null_pct_max"
]].head(12))
display(pd.DataFrame({
    "files": [len(inventory)],
    "bytes": [int(inventory.bytes.sum())],
    "rows": [int(inventory.rows.sum())],
    "elapsed_seconds": [round(audit_seconds, 2)],
    "peak_rss_mb_process": [round(peak_rss_mb, 1)],
}))

In [ ]:
display(Markdown(
    f"**Resultado.** Se leyeron completamente `{len(inventory)}` CSV con "
    f"`{int(inventory.rows.sum()):,}` filas y `{int(inventory.bytes.sum()):,}` bytes en "
    f"`{audit_seconds:.1f}` s. El máximo RSS observado por el proceso fue `{peak_rss_mb:.1f}` MB. "
    "El inventario se guardó en `artifacts/tables/raw_file_inventory.parquet`."
))

## Esquema, nulos y granularidad

Se compara el esquema por año y se audita la clave candidata `AELEC × EDOCVE × SECCION × SEXO × EDAD`. Esta clave expresa sección por sexo por edad, no casilla ni una fila agregada territorial. Los duplicados exactos se calculan por separado.

In [ ]:
summary_view = year_summary[[
    "year", "files", "rows", "schema_matches_expected", "sections", "municipalities",
    "duplicate_key_groups", "duplicate_key_extra_rows", "exact_duplicate_groups",
    "exact_duplicate_extra_rows", "ln_zero_rows", "identified_zero_rows"
]].copy()
display(summary_view)

null_matrix = pd.DataFrame(
    {int(row.year): row.null_counts for row in year_summary.itertuples(index=False)}
).fillna(0).astype("int64")
display(null_matrix)

In [ ]:
schema_ok = bool(year_summary.schema_matches_expected.all())
key_duplicates = int(year_summary.duplicate_key_extra_rows.sum())
exact_duplicates = int(year_summary.exact_duplicate_extra_rows.sum())
display(Markdown(
    f"**Hallazgo.** Los seis años comparten las 16 columnas esperadas: `{schema_ok}`. La clave "
    f"candidata sección × sexo × edad tiene `{key_duplicates:,}` filas adicionales en grupos "
    f"duplicados y existen `{exact_duplicates:,}` duplicados exactos. Las repeticiones territoriales "
    "entre sexo y edad son dimensiones sustantivas, no duplicados. Los nulos se concentran en los "
    "campos mostrados y no se imputan en esta fase."
))

## Categorías de sexo, edad y tipo de sección

Se muestran las categorías observadas. La ausencia de `SEXO=2` antes de 2024 se interpreta como categoría no disponible según la documentación, no como conteo cero. Los códigos de edad se conservan sin asignar todavía grupos abiertos.

In [ ]:
sex_categories = categories[categories.column.eq("SEXO")].copy()
section_type_categories = categories[categories.column.eq("TIPOSEC")].copy()
age_categories = categories[categories.column.eq("EDAD")].copy()
age_summary = (
    age_categories.assign(age_numeric=pd.to_numeric(age_categories.value, errors="coerce"))
    .groupby("year", as_index=False)
    .agg(
        category_count=("value", "nunique"),
        min_age=("age_numeric", "min"),
        max_age=("age_numeric", "max"),
        nonnumeric_categories=("age_numeric", lambda values: int(values.isna().sum())),
    )
)
age_outliers = (
    age_categories.assign(age_numeric=pd.to_numeric(age_categories.value, errors="coerce"))
    .loc[lambda frame: frame.age_numeric.gt(100)]
    .groupby("year", as_index=False)
    .agg(rows_over_100=("rows", "sum"), nominal_list_over_100=("nominal_list", "sum"), max_age=("age_numeric", "max"))
)
display(sex_categories)
display(section_type_categories)
display(age_summary)
display(age_outliers)

In [ ]:
non_binary_2024 = sex_categories[
    sex_categories.year.eq(2024) & sex_categories.value.astype(str).eq("2")
]
display(Markdown(
    f"**Hallazgo.** `SEXO=2` aparece en 2024 con `{int(non_binary_2024.rows.sum()):,}` filas y "
    f"lista nominal agregada `{int(non_binary_2024.nominal_list.sum()):,}`; no aparece en años "
    "anteriores. Los códigos de tipo de sección observados se reportan arriba. La edad llega al "
    f"rango `{int(age_summary.min_age.min())}`–`{int(age_summary.max_age.max())}`; el manual del "
    "tablero agrupa 100+, pero los TXT sólo describen años cumplidos, por lo que la semántica de "
    "edades mayores a 100 queda pendiente antes de agrupar o validar."
))

## Conciliación contable e indicadores

Se verifica fila por fila `LN = SV + NV + NS`. Después se agregan numeradores y denominadores antes de dividir. La participación oficial del tablero es `SV/(SV+NV)`; `SV/LN` se etiqueta sólo como sensibilidad. Si un denominador fuera cero, el resultado queda no definido (`NA`).

In [ ]:
counts = year_summary[[
    "year", "rows", "accounting_mismatch_rows", "accounting_max_abs_difference",
    "sum_ln", "sum_sv", "sum_nv", "sum_ns"
]].copy()
identified = counts.sum_sv + counts.sum_nv
counts["official_participation"] = counts.sum_sv.div(identified.where(identified.ne(0)))
counts["official_abstention"] = counts.sum_nv.div(identified.where(identified.ne(0)))
counts["unspecified_share_ln"] = counts.sum_ns.div(counts.sum_ln.where(counts.sum_ln.ne(0)))
counts["identified_coverage_ln"] = identified.div(counts.sum_ln.where(counts.sum_ln.ne(0)))
counts["sensitivity_voted_over_ln"] = counts.sum_sv.div(counts.sum_ln.where(counts.sum_ln.ne(0)))
display(counts)

In [ ]:
mismatch_total = int(counts.accounting_mismatch_rows.sum())
display(Markdown(
    f"**Resultado.** La identidad contable presenta `{mismatch_total:,}` filas discrepantes. "
    "Los porcentajes nacionales mostrados se calcularon como razones de sumas, no como promedio "
    "simple entre secciones. NS permanece fuera de abstención y se informa por separado sobre LN."
))

## Equivalencia de campos y ausencias del archivo descargable

Se construye una matriz conceptual. También se registran dimensiones del tablero que no aparecen como columnas en los CSV; su presencia en la plataforma no autoriza a fabricarlas ni a atribuirlas a personas.

In [ ]:
concepts = [
    ("año electoral", "AELEC"), ("fecha electoral", "FELECCION"),
    ("entidad", "EDOCVE / EDONOM"), ("municipio", "MPIOCVE / MPIONOM"),
    ("sección", "SECCION"), ("tipo de sección", "TIPOSEC"),
    ("distrito federal", "DEF"), ("distrito local", "DEL"),
    ("sexo", "SEXO"), ("edad", "EDAD"), ("lista nominal", "LN"),
    ("sí votaron", "SV"), ("no votaron", "NV"),
    ("registros no especificados", "NS"),
    ("porcentaje de participación publicado", None),
    ("referente indígena", None), ("referente afromexicano", None),
]
equivalence_rows = []
for concept, source_field in concepts:
    row = {"campo_conceptual": concept}
    for year in [2009, 2012, 2015, 2018, 2021, 2024]:
        row[str(year)] = source_field if source_field else "NO PRESENTE EN CSV"
    if concept == "sexo":
        row["discrepancia"] = "SEXO=2 sólo está documentado y observado en 2024"
    elif concept in {"distrito federal", "distrito local"}:
        row["discrepancia"] = "Marco DERFE 2021 en 2009–2021; marco DERFE 2023 en 2024"
    elif source_field is None:
        row["discrepancia"] = "Disponible o derivado en tablero, no como campo del ZIP general"
    else:
        row["discrepancia"] = "Nombre común; definición sujeta al diccionario de cada versión"
    equivalence_rows.append(row)
equivalence = pd.DataFrame(equivalence_rows)
equivalence_path = PROJECT_ROOT / "artifacts/tables/field_equivalence_matrix.csv"
equivalence.to_csv(equivalence_path, index=False)
display(equivalence)

In [ ]:
missing_dashboard_fields = equivalence[
    equivalence["2009"].eq("NO PRESENTE EN CSV")
].campo_conceptual.tolist()
display(Markdown(
    "**Hallazgo.** Los conteos LN/SV/NV/NS y las dimensiones básicas sí mantienen nombres de "
    "campo comunes. No aparecen columnas de " + ", ".join(missing_dashboard_fields) +
    ". Por tanto, los referentes indígena/afromexicano se tratan provisionalmente como atributos "
    "territoriales o enriquecimientos externos del tablero, nunca como identidad individual."
))

## Comparabilidad territorial

Se comparan identificadores de sección tal como vienen (`EDOCVE:SECCION`) y cambios de metadatos. La superposición de códigos mide coincidencia nominal, no equivalencia geométrica.

In [ ]:
section_frames = section_frames.assign(
    section_code=section_frames.EDOCVE.astype(str).str.zfill(2) + ":" + section_frames.SECCION.astype(str)
)
years = [2009, 2012, 2015, 2018, 2021, 2024]
section_sets = {
    year: set(section_frames.loc[section_frames.year.eq(year), "section_code"])
    for year in years
}
overlap_rows = []
for previous, current in zip(years[:-1], years[1:]):
    intersection = section_sets[previous] & section_sets[current]
    overlap_rows.append({
        "previous": previous,
        "current": current,
        "previous_sections": len(section_sets[previous]),
        "current_sections": len(section_sets[current]),
        "shared_codes": len(intersection),
        "only_previous": len(section_sets[previous] - section_sets[current]),
        "only_current": len(section_sets[current] - section_sets[previous]),
    })
overlap = pd.DataFrame(overlap_rows)

left = section_frames[section_frames.year.eq(2021)].set_index("section_code")
right = section_frames[section_frames.year.eq(2024)].set_index("section_code")
shared = left.join(right, how="inner", lsuffix="_2021", rsuffix="_2024")
metadata_changes = pd.DataFrame({
    "field": ["MPIOCVE", "DEF", "DEL", "TIPOSEC"],
    "changed_shared_codes": [
        int((shared[f"{field}_2021"].fillna("<NA>") != shared[f"{field}_2024"].fillna("<NA>")).sum())
        for field in ["MPIOCVE", "DEF", "DEL", "TIPOSEC"]
    ],
    "shared_codes": len(shared),
})
display(overlap)
display(metadata_changes)

In [ ]:
display(Markdown(
    f"**Interpretación.** Entre 2021 y 2024 coinciden nominalmente `{len(shared):,}` códigos de "
    "sección, pero existen altas/bajas y cambios de municipio, distrito o tipo de sección según la "
    "tabla. Esto confirma que repetir ID no basta para afirmar territorio idéntico. Hace falta "
    "cartografía oficial, catálogos y, si existe, un crosswalk autorizado antes de construir el panel."
))

## Reporte HTML y comprobaciones de cierre

El HTML de auditoría reúne las tablas obtenidas. Se verifican los artefactos mínimos; no se declara armonización, target ni reproducibilidad completa.

In [ ]:
report_path = PROJECT_ROOT / "artifacts/reports/raw_audit.html"
report_path.parent.mkdir(parents=True, exist_ok=True)
html_parts = [
    "<h1>RADAR 27 · Auditoría inicial de archivos crudos</h1>",
    "<p>Generado por notebooks/01_raw_schema_audit.ipynb. Código y contexto completo en la copia ejecutada.</p>",
    "<h2>Resumen por año</h2>", summary_view.to_html(index=False),
    "<h2>Conciliación e indicadores</h2>", counts.to_html(index=False),
    "<h2>Categorías de sexo</h2>", sex_categories.to_html(index=False),
    "<h2>Tipos de sección</h2>", section_type_categories.to_html(index=False),
    "<h2>Edades</h2>", age_summary.to_html(index=False),
    "<h2>Edades mayores a 100</h2>", age_outliers.to_html(index=False),
    "<h2>Equivalencia de campos</h2>", equivalence.to_html(index=False),
    "<h2>Superposición territorial nominal</h2>", overlap.to_html(index=False),
    "<h2>Cambios de metadatos 2021–2024</h2>", metadata_changes.to_html(index=False),
]
report_path.write_text("\n".join(html_parts), encoding="utf-8")

required_outputs = [
    PROJECT_ROOT / "artifacts/tables/raw_file_inventory.parquet",
    PROJECT_ROOT / "artifacts/tables/field_equivalence_matrix.csv",
    PROJECT_ROOT / "artifacts/tables/category_inventory.csv",
    report_path,
]
checks = {
    "all_192_csv_audited": len(inventory) == 192,
    "six_years_audited": len(year_summary) == 6,
    "all_outputs_exist": all(path.exists() for path in required_outputs),
    "accounting_identity_holds": int(counts.accounting_mismatch_rows.sum()) == 0,
    "no_exact_duplicates": int(year_summary.exact_duplicate_extra_rows.sum()) == 0,
    "cartography_compatibility_resolved": False,
}
display(pd.Series(checks, name="aprobado").to_frame())
elapsed = time.perf_counter() - RUN_STARTED
display(Markdown(
    f"**Cierre.** Pasaron `{sum(checks.values())}/{len(checks)}` comprobaciones en `{elapsed:.1f}` s. "
    "La comprobación territorial falla deliberadamente porque aún no hay geometría/crosswalk validado. "
    "Siguiente paso concreto: definir reglas de limpieza sin recodificar categorías, obtener catálogos "
    "cartográficos oficiales y probar reconciliación territorial antes del panel."
))